# LAB 06 — Bài 16 & 17: Scaled Dot-Product Attention

Notebook này triển khai attention bằng NumPy và softmax ổn định số. Không dùng `torch.nn.MultiheadAttention` hoặc `keras.layers.Attention`.

## Bài 16 — Core implementation

Với `Q (..., Tq, dk)`, `K (..., Tk, dk)`, `V (..., Tk, dv)`, tính:

1. `scores = Q @ K.T / sqrt(dk)`
2. `attention_weights = softmax(scores)` theo trục key (`Tk`)
3. `context = attention_weights @ V`

Hàm trả về `(scores, attention_weights, context)`. Có hỗ trợ batch dimensions theo quy tắc broadcasting của NumPy.

## Bài 17 — Numerical stability

Softmax trừ đi giá trị lớn nhất trước khi lấy `exp`, để hạn chế overflow với score lớn. Cell cuối thử vector `[1000, 1001, 1002]`.


In [ ]:
import numpy as np


def stable_softmax(x, axis=-1):
    """Softmax ổn định số trên trục axis."""
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        raise ValueError("x must not be empty")

    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    """
    Scaled dot-product attention thuần NumPy.

    Shapes:
        Q: (..., Tq, dk)
        K: (..., Tk, dk)
        V: (..., Tk, dv)

    Returns:
        scores:            (..., Tq, Tk), đã chia sqrt(dk)
        attention_weights:(..., Tq, Tk), tổng mỗi hàng bằng 1
        context:           (..., Tq, dv)
    """
    Q = np.asarray(Q, dtype=np.float64)
    K = np.asarray(K, dtype=np.float64)
    V = np.asarray(V, dtype=np.float64)

    if Q.ndim < 2 or K.ndim < 2 or V.ndim < 2:
        raise ValueError("Q, K, V must each have at least 2 dimensions")
    if Q.shape[-1] != K.shape[-1]:
        raise ValueError("Q and K must have the same key dimension")
    if K.shape[-2] != V.shape[-2]:
        raise ValueError("K and V must have the same sequence length")
    if Q.shape[-1] == 0:
        raise ValueError("key dimension dk must be positive")

    d_k = Q.shape[-1]
    scores = np.matmul(Q, np.swapaxes(K, -1, -2)) / np.sqrt(d_k)
    attention_weights = stable_softmax(scores, axis=-1)
    context = np.matmul(attention_weights, V)

    return scores, attention_weights, context

In [ ]:
# Bài 17: thử softmax với score lớn
x = np.array([1000.0, 1001.0, 1002.0])
probabilities = stable_softmax(x)
print("Input scores:", x)
print("Softmax:", probabilities)
print("Sum of weights:", probabilities.sum())

# Ví dụ dữ liệu nhỏ để xem đủ scores, attention weights và context.
# Thay Q, K, V bằng dữ liệu của bạn khi làm các câu tính tay.
Q = np.array([[0.5, -0.2]])
K = np.array([[0.1, 0.4], [-0.3, 0.8], [0.7, 0.2]])
V = np.array([[0.2, 0.5], [1.2, -0.1], [-0.4, 0.7]])

scores, attention_weights, context = scaled_dot_product_attention(Q, K, V)
print("\nScaled scores:\n", scores)
print("Attention weights:\n", attention_weights)
print("Context:\n", context)
print("Row sums:", attention_weights.sum(axis=-1))

Input scores: [1000. 1001. 1002.]
Softmax: [0.09003057 0.24472847 0.66524096]
Sum of weights: 0.9999999999999999

Scaled scores:
 [[-0.0212132 -0.2192031  0.2192031]]
Attention weights:
 [[0.32339888 0.26530937 0.41129174]]
Context:
 [[0.21853433 0.42307272]]
Row sums: [1.]
